## 6. Urban rural segregation.
It categorizes buildings based on their location into a fitting urbanization status, uses the pre-processed polygons from previous notebook - 5_rural_urban_json_segregation.

### Initial configuration

    config_str = """{"REGION": "Patan",
    "COS_ENDPOINT_URL": "https://s3.direct.eu-de.cloud-object-storage.appdomain.cloud",
    "COS_AUTH_ENDPOINT_URL":"https://iam.cloud.ibm.com/oidc/token",
    "COS_APIKEY": "xxx",
    "UTILS_BUCKET": "notebook-utils-bucket",
    "VIDA_COUNTRIES_BUILDINGS": "onboarding-bucket-j"}"""

* `REGION`: Region of study.
* `COS_AUTH_ENDPOINT_URL`: The authentication endpoint for generating IBM IAM tokens.
* `COS_APIKEY`: Your unique IBM Cloud API key for programmatic access.
* `UTILS_BUCKET`: Where the .py file to download the package `filtering_grid_generator`.
* `VIDA_COUNTRIES_BUILDINGS`: Bucket to upload results.


In [1]:
# Import necessary libraries
import pandas as pd
import geopandas as gpd
import jaydebeapi as jdbc
import jpype
import os
import shapely
import rasterio as rio
from rasterio.plot import show
from rasterio.mask import mask
import matplotlib.pyplot as plt
import  rioxarray
from skimage import measure as M
import requests
from botocore.client import Config
import ibm_boto3
import io
import getpass
import json
from tqdm import tqdm
from datetime import datetime

In [2]:
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

'2025-10-03 15:02:34'

In [4]:
#config_str = getpass.getpass('Enter your prepared config: ')
config = json.loads(config_str)

Dictioaries with the classification type are defined:

In [5]:
# Overview names
segregation_overview_names = {
    'URBAN': 'Urban',
    'SUBURBAN': 'Suburban',
    'RURAL': 'Rural'
}

# Detailed names
segregation_detailed_names = {
    'URBAN_CENTER': 'Urban Center',
    'DENSE_URBAN': 'Dense Urban Cluster',
    'SEMI_DENSE_URBAN': 'Semi-dense Urban Cluster',
    'SUBURBAN_PERI_URBAN': 'Suburban or Peri-urban',
    'RURAL_CLUSTER': 'Rural Cluster',
    'LOW_DENSITY_RURAL': 'Low Density Rural'
}

Conection to the IBM cloud is established:

In [6]:
#load db2jcc4.jar and polygons json
cos_client = ibm_boto3.client(service_name='s3',
                              ibm_api_key_id=config["COS_APIKEY"],
                              config=Config(signature_version='oauth'),
                              endpoint_url=config["COS_ENDPOINT_URL"])

In [7]:
output_folder = "/home/julian/notebooks/"

`parquet_file` referst to the parquet from one of the previous notebook [4. Building height calculation.](https://10.243.64.5/user/julian/lab/tree/handover/notebooks/4_building_height_calculation_parquet.ipynb) where height per building was computed.

In [8]:
parquet_file = f"{config["REGION"]}_height_NEW.parquet"
parquet_file_path = os.path.join(output_folder, parquet_file)

Coneection to a resource of IBM. Iterating trhough the files and getting the ones from the previous notebook [5. Urban-Rural Polygon Generation from GHS-SMOD](https://10.243.64.5/user/julian/lab/tree/handover/notebooks/5_rural_urban_json_segregation.ipynb)

In [9]:
# Connect to COS as a resource (not client!)
cos_resource = ibm_boto3.resource(service_name='s3',
                                  ibm_api_key_id=config["COS_APIKEY"],
                                  config=Config(signature_version='oauth'),
                                  endpoint_url=config["COS_ENDPOINT_URL"])

parquet_bucket = cos_resource.Bucket(config["OUTPUT_BUCKET"])

# get all filenames from the parquet bucket
all_json_files = [i.key for i in parquet_bucket.objects.all()]

state = config["REGION"]
smod_json_all = [i for i in all_json_files if (state+'_' in i) and ('.json' in i)]
parquet_file =[i for i in all_json_files if (parquet_file_path in i)]

Once the files are identified, they are locally stored for post-processing:

In [10]:
# Fetch the geoJSON file containing polygons and process each polygon one by one
for smod_json in smod_json_all:
    
    streaming_body = cos_client.get_object(Bucket=config["OUTPUT_BUCKET"], Key=smod_json)['Body']
    print("Downloading to local storage :  " + smod_json)
    with io.FileIO(smod_json, 'w') as file:
        for i in io.BytesIO(streaming_body.read()):
            file.write(i)

# Fetch the geoJSON file containing polygons and process each polygon one by one
for vida_buildings_parquet in parquet_file:
    
    #streaming_body = cos_client.get_object(Bucket=config["VIDA_COUNTRIES_BUILDINGS"], Key=vida_buildings_parquet)['Body']
    streaming_body = cos_client.get_object(Bucket=config["OUTPUT_BUCKET"], Key=vida_buildings_parquet)['Body']
    print("Downloading to local storage :  "+vida_buildings_parquet)
    with io.FileIO(vida_buildings_parquet, 'w') as file:
        for i in io.BytesIO(streaming_body.read()):
            file.write(i)



In [11]:
smod_overview_json_filenames = [f'{config['REGION']}_overview.json']
smod_overview_json_filenames.sort()

smod_detailed_json_filenames = [f'{config['REGION']}_detailed.json']
smod_detailed_json_filenames.sort()

vida_datasets = [parquet_file_path]
vida_datasets.sort()
#smod_overview_json_filenames, smod_detailed_json_filenames, vida_datasets

This final part of the script performs a two-stage geospatial analysis to classify buildings based on their location within different types of geographical polygons. It first assigns a broad urban/rural category and then refines it with a more detailed classification. Finally, it saves the augmented data and uploads it to cloud storage.

In [12]:
for smod_json_overview, smod_json_detailed, parquet_filename in zip(smod_overview_json_filenames, smod_detailed_json_filenames, vida_datasets):
#Open Kenya_segregated polygons

     # === OVERVIEW PROCESSING ===
    with open(smod_json_overview) as s_f:
        geojson_overview = json.load(s_f)

    print('Processing: ', smod_json_overview, smod_json_detailed, parquet_filename)

    buildings_df = pd.read_parquet(parquet_filename)
    init_len = len(buildings_df)
    dfs = []
    
    for fidx, feature in enumerate(tqdm(geojson_overview['features'], total=len(geojson_overview['features']), desc='Processing overview level')):
        polygon_coordinates = feature['geometry']['coordinates'][0]
        polygon = shapely.Polygon(polygon_coordinates)
        min_lon, min_lat, max_lon, max_lat = polygon.bounds
        
        buildings_bbox = buildings_df[
                (buildings_df.longitude >= min_lon) &
                (buildings_df.longitude <= max_lon) &
                (buildings_df.latitude >= min_lat) &
                (buildings_df.latitude <= max_lat)
        ].copy()
        
        buildings_bbox['buildings_in_polygon'] = [polygon.contains(shapely.Point(row.longitude, row.latitude)) for row in buildings_bbox.itertuples()]

        buildings_in_polygon = buildings_bbox[buildings_bbox.buildings_in_polygon == True]        
        
        buildings_in_polygon = buildings_in_polygon.drop(columns=['buildings_in_polygon'])
        buildings_in_polygon['urban_split'] = segregation_overview_names [feature['properties']['seg_type']]
        
        dfs.append(buildings_in_polygon)
     
    try: 
        segregated_df = pd.concat(dfs)
        
        remaining_buildings = buildings_df[~buildings_df['id'].isin(segregated_df['id'])]
        remaining_buildings['urban_split'] = 'Rural'
        
        main_df = pd.concat([segregated_df, remaining_buildings])
        
        print(f'init len {init_len} out len {len(main_df)}')
    except Exception as e:
        print(e)
        
    #################################################################################################################
    print('removing duplicates')
    main_df = main_df.drop_duplicates(subset='id')
    print(f'in len {init_len} out len {len(main_df)}')

    main_df.index = [i for i in range(len(main_df))]

     # === DETAILED PROCESSING ===
    with open(smod_json_detailed) as s_f:
        geojson_detailed = json.load(s_f)
        
    dfs = []
        
    for fidx, feature in enumerate(tqdm(geojson_detailed['features'], total=len(geojson_detailed['features']), desc='Processing detailed level')):
        
        polygon_coordinates = feature['geometry']['coordinates'][0]
        polygon = shapely.Polygon(polygon_coordinates)
        min_lon, min_lat, max_lon, max_lat = polygon.bounds
        
        buildings_bbox = main_df[
                (main_df.longitude >= min_lon) &
                (main_df.longitude <= max_lon) &
                (main_df.latitude >= min_lat) &
                (main_df.latitude <= max_lat)
        ].copy()
        
        buildings_bbox['buildings_in_polygon'] = [polygon.contains(shapely.Point(row.longitude, row.latitude)) for row in buildings_bbox.itertuples()]

        buildings_in_polygon = buildings_bbox[buildings_bbox.buildings_in_polygon == True]        
        
        buildings_in_polygon = buildings_in_polygon.drop(columns=['buildings_in_polygon'])
        buildings_in_polygon['ghsl_smod'] = segregation_detailed_names[feature['properties']['seg_type']]
        
        dfs.append(buildings_in_polygon)
     
    try: 
        segregated_df = pd.concat(dfs)
        
        remaining_buildings = main_df[~main_df['id'].isin(segregated_df['id'])]
        remaining_buildings['ghsl_smod'] = 'Low Density Rural'
        
        main_df = pd.concat([segregated_df, remaining_buildings])
        
        main_df = main_df.drop_duplicates(subset='id')
        
        print(f'in len {init_len} out len {len(main_df)}')
    except Exception as e:
        print(e)
        
    try:
        
        filename = parquet_filename.replace('.parquet', '_GHSL_SMOD.parquet')
        main_df.to_parquet(filename)
        
        res=cos_client.upload_file(Filename=filename, Bucket=config["VIDA_COUNTRIES_BUILDINGS"],Key=filename)
        print(f'File uploaded to the COS {filename}')
    except Exception as e:
        print(e)

Processing:  Patan_overview.json Patan_detailed.json /home/julian/notebooks/Patan_height_NEW.parquet


Processing overview level: 100%|██████████| 104/104 [00:05<00:00, 17.62it/s]
/tmp/ipykernel_9100/2430570257.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  remaining_buildings['urban_split'] = 'Rural'


init len 182313 out len 282970
removing duplicates
in len 182313 out len 182313


Processing detailed level: 100%|██████████| 338/338 [00:12<00:00, 26.14it/s]
/tmp/ipykernel_9100/2430570257.py:86: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  remaining_buildings['ghsl_smod'] = 'Low Density Rural'


in len 182313 out len 182313
'VIDA_COUNTRIES_BUILDINGS'


In [13]:
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

'2025-10-03 15:02:55'